# TrafficVision — Entrenamiento YOLO11n
**Tesis:** Detección y lectura de placas vehiculares ecuatorianas  
**Modelo:** YOLO11n (nano) — optimizado para Colab Free T4  
**Dataset:** 7,576 imágenes combinadas (global + Ecuador)

---
### 📋 Orden de ejecución
| Celda | Descripción | Obligatoria |
|-------|-------------|-------------|
| 0 | Anti-desconexión | ✅ Siempre |
| 1 | Verificar GPU | ✅ Siempre |
| 2 | Instalar dependencias | ✅ Siempre |
| 3 | Montar Drive | ✅ Siempre |
| 4 | Verificar datasets | ✅ Siempre |
| 5 | Crear YAML | ✅ Siempre |
| 6 | **Entrenar** (nuevo) | 🔵 Primera vez |
| 7 | **Reanudar** (interrumpido) | 🟡 Si se cortó |
| 8 | Evaluar métricas | ✅ Al finalizar |
| 9 | Exportar modelo | ✅ Al finalizar |

In [ ]:
# ══════════════════════════════════════════════════════════════════
# CELDA 0 — Anti-desconexión + monitor de sesión
# ══════════════════════════════════════════════════════════════════
import time, threading

def heartbeat():
    """Evita la desconexión automática de Colab cada 90 min."""
    clicks = 0
    while True:
        time.sleep(45)
        clicks += 1
        try:
            from google.colab import output
            output.eval_js('document.querySelector("#top-toolbar").click()')
        except Exception:
            pass

t = threading.Thread(target=heartbeat, daemon=True)
t.start()

SESSION_START = time.time()
print('Anti-desconexión activo')
print('Sesión iniciada. Colab Free permite ~4-5h de GPU continua.')
print('Si se interrumpe, usa CELDA 7 (Reanudar) — no pierdas el progreso.')

In [ ]:
# CELDA 1 — Verificar GPU y RAM disponible
!nvidia-smi

import torch, psutil, os

# GPU
cuda_ok = torch.cuda.is_available()
print(f'\n🔧 CUDA disponible: {cuda_ok}')
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'   GPU:  {gpu_name}')
    print(f'   VRAM: {gpu_mem:.1f} GB')
    # Recomendación de batch según VRAM
    if gpu_mem >= 14:
        rec_batch = 16
    elif gpu_mem >= 8:
        rec_batch = 8
    else:
        rec_batch = 4
    print(f'Batch recomendado: {rec_batch} (para imgsz=640)')
else:
    print(' Sin GPU — el entrenamiento será muy lento en CPU.')
    print(' Solución: Runtime → Cambiar tipo de entorno de ejecución → T4 GPU')

# RAM del sistema
ram = psutil.virtual_memory()
print(f'\n RAM sistema: {ram.available/1024**3:.1f} GB disponibles / {ram.total/1024**3:.1f} GB total')

# Espacio en disco
disk = psutil.disk_usage('/')
print(f' Disco /tmp:   {disk.free/1024**3:.1f} GB libres')

In [ ]:
# CELDA 2 — Instalar dependencias
# ultralytics >= 8.3 incluye soporte completo para YOLO11
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics
print(f'✅ Ultralytics {ultralytics.__version__} instalado (soporta YOLO11)')

# Verificar versión mínima
major, minor = map(int, ultralytics.__version__.split('.')[:2])
if major < 8 or (major == 8 and minor < 3):
    print('⚠️  Versión antigua — puede no soportar YOLO11. Reinicia el runtime.')
else:
    print(f'   ✅ Versión compatible con YOLO11')

In [ ]:
# CELDA 3 — Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Rutas base del proyecto
DRIVE_BASE  = '/content/drive/MyDrive/TrafficVision/datasets'
DRIVE_RUNS  = '/content/drive/MyDrive/TrafficVision/runs'
RUN_NAME    = 'yolo11n_combined_all'

# Crear carpeta de runs si no existe
import os
os.makedirs(DRIVE_RUNS, exist_ok=True)

print('✅ Google Drive montado')
print(f'   Datasets: {DRIVE_BASE}')
print(f'   Runs:     {DRIVE_RUNS}')

In [ ]:
# CELDA 4 — Verificar datasets y estimar tiempo de entrenamiento
import os

datasets = {
    'license-plates (global)':  f'{DRIVE_BASE}/license-plates',
    'license-plates-ec-1':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1',
    'license-plates-ec-2':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2',
    'license-plates-ec-4':      f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4',
}

total_train = 0
total_val   = 0
all_ok      = True

print('  VERIFICACIÓN DE DATASETS')

for name, path in datasets.items():
    exists = os.path.exists(path)
    if exists:
        train_path = f'{path}/train/images'
        val_path   = f'{path}/valid/images'
        n_train = len(os.listdir(train_path)) if os.path.exists(train_path) else 0
        n_val   = len(os.listdir(val_path))   if os.path.exists(val_path)   else 0
        total_train += n_train
        total_val   += n_val
        print(f'  ✅ {name}')
        print(f'     train: {n_train:,} imgs  |  val: {n_val:,} imgs')
    else:
        all_ok = False
        print(f'  ❌ {name} — NO ENCONTRADO')
        print(f'     Ruta esperada: {path}')

print(f'  TOTAL train: {total_train:,} imágenes')
print(f'  TOTAL val:   {total_val:,} imágenes')

# Estimación de tiempo (T4, batch=16, imgsz=640)
# ~2.5 seg/epoch por cada 1000 imgs en T4
secs_per_epoch = (total_train / 1000) * 2.5
total_mins     = (secs_per_epoch * 100) / 60
print(f'\n  Estimación para 100 épocas en T4 (batch=16):')
print(f'   ~{secs_per_epoch:.0f} seg/época  →  ~{total_mins:.0f} min totales ({total_mins/60:.1f} h)')
print(f'   Colab Free: máx ~4-5h por sesión.')

if total_mins > 240:
    safe_epochs = int((240 * 60) / secs_per_epoch)
    print(f'   Con este dataset, una sesión alcanza ~{safe_epochs} épocas.')
    print(f'   Usa save_period=5 y reanuda con CELDA 7 en la siguiente sesión.')

if not all_ok:
    print('\n Algunos datasets faltan. Verifica que estén en Drive antes de entrenar.')

In [ ]:
# CELDA 5 — Crear data_combined_all.yaml
import yaml, os

data = {
    'train': [
        f'{DRIVE_BASE}/license-plates/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-1/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-2/train/images',
        f'{DRIVE_BASE}/license-plates-ec-combined/license-plates-ec-4/train/images',
    ],
    # Validación solo con el dataset global (más imágenes = métricas más confiables)
    'val':  f'{DRIVE_BASE}/license-plates/valid/images',
    'test': f'{DRIVE_BASE}/license-plates/test/images',
    'nc':   1,
    'names': ['license plate'],
}

YAML_PATH = '/content/data_combined_all.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(data, f, default_flow_style=False, allow_unicode=True)

print('✅ data_combined_all.yaml creado')
print(f'   Ruta: {YAML_PATH}')
print(f'   Clases: {data["nc"]} ({data["names"]})')
print(f'   Carpetas train: {len(data["train"])}')
for p in data['train']:
    n = len(os.listdir(p)) if os.path.exists(p) else '❌ no existe'
    label = '/'.join(p.split('/')[-4:-2])
    print(f'     {label}: {n} imgs')

# Verificar labels también
print('\n  Verificando labels (.txt)...')
for p in data['train']:
    lp = p.replace('/images', '/labels')
    if os.path.exists(lp):
        n = len([f for f in os.listdir(lp) if f.endswith('.txt')])
        label = '/'.join(p.split('/')[-4:-2])
        print(f'     ✅ {label}: {n} labels')
    else:
        print(f'     Labels no encontrados en {lp}')

In [ ]:
# CELDA 6 — ENTRENAR YOLO11n (primera vez)
# ⚠️  Solo ejecutar si NO existe un checkpoint previo.
#     Si el entrenamiento se interrumpió → usa CELDA 7 (Reanudar).
import os, time
from ultralytics import YOLO

# Verificar que no haya checkpoint previo accidentalmente
checkpoint = f'{DRIVE_RUNS}/{RUN_NAME}/weights/last.pt'
if os.path.exists(checkpoint):
    size_mb = os.path.getsize(checkpoint) / 1024**2
    print(f'⚠️  Ya existe un checkpoint: {checkpoint} ({size_mb:.1f} MB)')
    print('   Si quieres REANUDAR → usa CELDA 7')
    print('   Si quieres empezar DE CERO → cambia RUN_NAME arriba o borra la carpeta')
    print('   Deteniendo para no sobreescribir...')
    raise SystemExit('Checkpoint existente — usa CELDA 7 para reanudar.')

print('🚀 Iniciando entrenamiento YOLO11n...')
print(f'   Dataset:  {YAML_PATH}')
print(f'   Destino:  {DRIVE_RUNS}/{RUN_NAME}')
print()

model = YOLO('yolo11n.pt')   # Nano — más ligero, suficiente para detección de 1 clase

# ─── Parámetros optimizados para Colab Free T4 ────────────────────
# batch=16:       Balance seguro VRAM/velocidad en T4 (14.9 GB)
# imgsz=640:      Estándar YOLO, no reducir (perdería precisión en placas pequeñas)
# cache='disk':   Evita recargar imágenes de Drive en cada época (crítico para Drive)
# save_period=5:  Guarda checkpoint cada 5 épocas → recuperación granular
# workers=2:      Drive es lento; más workers no ayudan y gastan RAM
# patience=20:    Early stopping más permisivo (dataset mixto puede tener ruido)
# cos_lr=True:    Convergencia más suave, mejor para datasets pequeños mezclados
# amp=True:       Mixed precision → 30% más rápido, menos VRAM
# ──────────────────────────────────────────────────────────────────

results = model.train(
    data          = YAML_PATH,
    epochs        = 100,
    imgsz         = 640,
    batch         = 16,          # ← seguro para T4; si sale OOM bajar a 8
    name          = RUN_NAME,
    project       = DRIVE_RUNS,
    patience      = 20,
    save          = True,
    save_period   = 5,           # ← checkpoint cada 5 épocas
    plots         = True,
    device        = 0,
    amp           = True,
    cos_lr        = True,
    cache         = 'disk',      # ← evita re-leer desde Drive en cada época
    workers       = 2,           # ← Drive es el cuello de botella, no la CPU
    warmup_epochs = 3,
    resume        = False,
    verbose       = True,
)

elapsed = (time.time() - SESSION_START) / 60
print(f'\n Entrenamiento completado en {elapsed:.1f} min')
print(f'   Modelo guardado en: {DRIVE_RUNS}/{RUN_NAME}/weights/best.pt')

In [ ]:
# CELDA 7 — REANUDAR entrenamiento interrumpido
# Usar cuando se desconectó o se agotó el tiempo de GPU.
# Ejecuta celdas 0-5 antes!

import glob, os
from ultralytics import YOLO

# Buscar el último checkpoint
last_pt = f'{DRIVE_RUNS}/{RUN_NAME}/weights/last.pt'

if not os.path.exists(last_pt):
    # Buscar cualquier checkpoint de este run
    candidates = glob.glob(f'{DRIVE_RUNS}/{RUN_NAME}/weights/*.pt')
    if candidates:
        last_pt = sorted(candidates)[-1]  # el más reciente
        print(f'  last.pt no encontrado, usando: {last_pt}')
    else:
        print(f'❌ No hay ningún checkpoint en {DRIVE_RUNS}/{RUN_NAME}/')
        print('   Ejecuta CELDA 6 para comenzar desde cero.')
        raise FileNotFoundError('Sin checkpoint para reanudar.')

size_mb = os.path.getsize(last_pt) / 1024**2
print(f'✅ Checkpoint encontrado: {last_pt}')
print(f'   Tamaño: {size_mb:.1f} MB')

# Verificar qué época tiene guardada
try:
    import torch
    ckpt = torch.load(last_pt, map_location='cpu', weights_only=False)
    epoch = ckpt.get('epoch', '?')
    print(f'   Última época guardada: {epoch}/100')
    del ckpt
except Exception:
    print('   (No se pudo leer la época del checkpoint)')

print('\n🔁 Reanudando entrenamiento...')

model = YOLO(last_pt)
results = model.train(
    data        = YAML_PATH,
    epochs      = 100,
    imgsz       = 640,
    batch       = 16,
    name        = RUN_NAME,
    project     = DRIVE_RUNS,
    patience    = 20,
    save        = True,
    save_period = 5,
    plots       = True,
    device      = 0,
    amp         = True,
    cos_lr      = True,
    cache       = 'disk',
    workers     = 2,
    resume      = True,         # ← clave: reanuda desde last.pt
    verbose     = True,
)

print('\n✅ Entrenamiento reanudado y completado')

In [ ]:
# CELDA 8 — Evaluar métricas del modelo entrenado
import glob, os
from ultralytics import YOLO

# Buscar best.pt
best_pt = f'{DRIVE_RUNS}/{RUN_NAME}/weights/best.pt'

if not os.path.exists(best_pt):
    print(f'❌ No se encontró best.pt en {DRIVE_RUNS}/{RUN_NAME}/')
    print('   El entrenamiento debe completarse (o llegar a al menos 1 época de validación).')
else:
    size_mb = os.path.getsize(best_pt) / 1024**2
    print(f'✅ Evaluando: {best_pt} ({size_mb:.1f} MB)')

    model = YOLO(best_pt)

    # Evaluar en validación
    metrics = model.val(
        data   = YAML_PATH,
        imgsz  = 640,
        device = 0,
        batch  = 16,
        plots  = True,
        save_json = False,
    )

    print()
    print('  MÉTRICAS FINALES — YOLO11n')
    print(f'  mAP@50:       {metrics.box.map50:.4f}  ({metrics.box.map50*100:.1f}%)')
    print(f'  mAP@50-95:    {metrics.box.map:.4f}  ({metrics.box.map*100:.1f}%)')
    print(f'  Precisión:    {metrics.box.mp:.4f}  ({metrics.box.mp*100:.1f}%)')
    print(f'  Recall:       {metrics.box.mr:.4f}  ({metrics.box.mr*100:.1f}%)')

    # Referencia vs modelo anterior
    map50 = metrics.box.map50
    prev  = 0.974  # YOLOv8n anterior
    delta = (map50 - prev) * 100
    icon  = '📈' if delta >= 0 else '📉'
    print(f'\n{icon} vs YOLOv8n anterior (97.4%): {delta:+.1f} puntos porcentuales')

    if map50 >= 0.95:
        print('  ✅ Excelente — listo para producción')
    elif map50 >= 0.85:
        print('  ⚠️  Aceptable — considera más épocas o fine-tuning en Ecuador')
    else:
        print('  ❌ Por debajo del umbral — revisa los datos o aumenta épocas')

In [ ]:
# CELDA 9 — Exportar modelo para producción
# Genera una copia en /content/ (RAM local) para descarga inmediata
# Y deja best.pt en Drive para uso en el backend.
import shutil, os
from ultralytics import YOLO

best_pt   = f'{DRIVE_RUNS}/{RUN_NAME}/weights/best.pt'
export_pt = f'/content/yolo11n_trafficvision_best.pt'

if not os.path.exists(best_pt):
    print(f'❌ No se encontró {best_pt}')
else:
    # Copiar a /content para descarga rápida
    shutil.copy2(best_pt, export_pt)
    size_mb = os.path.getsize(export_pt) / 1024**2
    print(f'✅ Modelo copiado a: {export_pt} ({size_mb:.1f} MB)')

    # Descarga directa desde Colab
    from google.colab import files
    print('\n📥 Iniciando descarga del modelo...')
    files.download(export_pt)

    print('\n📁 Ruta permanente en Drive:')
    print(f'   {best_pt}')
    print('\n🔧 Para usar en el backend (plate_detector.py):')
    print('   MODEL_PATH = "ml/models/trained/yolo11n_combined_all/best.pt"')

    # Resumen final del run
    results_csv = f'{DRIVE_RUNS}/{RUN_NAME}/results.csv'
    if os.path.exists(results_csv):
        import pandas as pd
        df = pd.read_csv(results_csv)
        df.columns = df.columns.str.strip()
        best_row = df.loc[df['metrics/mAP50(B)'].idxmax()]
        best_epoch = int(best_row['epoch']) + 1
        best_map50 = best_row['metrics/mAP50(B)']
        print(f'\n📊 Mejor época: {best_epoch}/100  →  mAP@50 = {best_map50:.4f} ({best_map50*100:.1f}%)')

---
## 💡 Guía rápida — Colab Free

### Si se desconecta durante el entrenamiento
1. Abre el notebook de nuevo
2. Ejecuta celdas **0 → 5** (anti-disco, GPU, instalar, Drive, verificar, YAML)
3. Ejecuta **CELDA 7** (Reanudar) — YOLO retoma desde el último `save_period`

### Señales de que va bien
- `box_loss` y `cls_loss` bajando cada época ✅
- `mAP50` subiendo progresivamente ✅  
- GPU mem ~4-6 GB (con batch=16) ✅

### Si sale `CUDA out of memory`
- Bajar `batch` de 16 → 8 en celdas 6 y 7
- Si persiste: `batch=4, imgsz=416`

### Tiempos estimados (T4, batch=16)
| Épocas | Tiempo estimado |
|--------|----------------|
| 30     | ~55 min        |
| 50     | ~92 min        |
| 100    | ~3 h           |

### Advertencia sobre segmentos
El WARNING `len(segments) != len(boxes)` en el dataset ec-1 es inofensivo —  
YOLO ignora las anotaciones de segmentación y usa solo los bounding boxes.
